# Problem Statement

The objective of this project is to develop a Machine Learning-based Network Intrusion Detection System using K-Nearest Neighbors (KNN) algorithm on the UNSW-NB15 dataset. The model aims to classify network traffic as normal or malicious.

### PHASE 1: PROBLEM UNDERTSNDING

Before touching any code, we need to undertand:
1. What is Network Intrusion Detection System?
2. What is the UNSW-NB15 dataset?
3. What are we trying to predict?
4. Why KNN can be used?
5. What will be our final project objective?


### Step 1.1: What is Intrusion Detection System (IDS)?


A cybersecurity tool that monitors network traffic or system activities to detect malicious actions, unauthorized access, or policy violations.

Imagine a company network.
Thousands of packets travel through the network every second.

some traffic is normal:
- Browsing websites
- Downloading files
- Sending emails

Some traffic is malicious:
- Hacking attempts
- DoS attacks
- Malware communication
- Unauthorized access attempts

An IDS tries to classify each network connection as:
- Normnal
- Attack

### Step 1.2: What is the UNSW-NB15 Dataset?

UNSW-NB15 is a modern cybersecurity dataset created by:

University of New South Wales.

It was develpoed beacuse older datasets such as:
- KDD Cup 99 Dataset
- NSL-KDD Dataset

became outdated.

#### What does UNSW-NB15 Contain?

Each row represents a network flow/connection.

Each row contains:
- Network characterstics
- Traffic statistics
- Protocol information
- Source/Destination information

and a target value telling whether the connection is malicious or Not.


### Step 1.3: Understanding the Target Columns

The dataset usually contains:

**label**

Binary classification:

0 -> Normal

1 -> Attack



**attack_cat**

Multi-class classification

Examples:
- Exploits
- DoS
- Fuzzers
- Worms
- Analysis
- Shellcode and so on.

### Which target should we use?


There are two possibilities:

**Option 1: label**

predict:

Normal vs Attack

This is called binary classification.

**Option 2: attack_cat**

Predict attack type.

This is called Multi-Class Classification.

I'm using **label** column as target variable as this is my first ML Project.

Beacuse:
- Easier preprocessing
- Easier evaluation
- Easier interpretation
- Easier KNN implementation
- Excellent for learning ML fundamentals

Later I can try with **attack_cat** as my target variable as an advanced version

### Step 1.4: What Type of ML Problemn Is This?

We have:

- Input features (X)
- Target (y)

Therefore:

This is a Supervised Machine Learning Classification Problem

### Step 1.5: Why KNN?



KNN works by:
1. Looking at nearby observations.
2. Finding the K closet neighbors.
3. Taking a vote.

Suppose: K = 5

Nearest neighbors:
- attack
- attack
- attack
- normal
- attack


Prediction: Attack

Beacause majority vote = attack

### Step 1.6: Final Project Goal

Our goal will be:

Build a KNN-based Intrusion Detection System that can classify a network connection as Normal vs Attack using the UNSW-NB15 dataset.

## Phase 2: Dataset Loading

The UNSW-NB15 dataset is loaded into a Pandas DataFrame for analysis. Pandas provides efficient data structures for handling large datasets and supports preprocessing operations required for machine learning.

Before loading the dataset we need to import modules that are required.

In [1]:
import pandas as pd
import numpy as np
import sklearn

### Step 2.1: Load a single Dataset File First

In [2]:
df1 = pd.read_csv(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\UNSW-NB15_1.csv",
                 header = None, low_memory = False)

Why **r** Before path?

The **r** means Raw string.

Without it: **\U or \n** are treated as escape characters in python.

Why **header = None** ?

The file donot contain column names. Without **header=None** Pandas assume the first row is a header row and may incorrectly remove actual data.

Why **low_memory = False** ?

Internally, pandas reads large files in chunks. With **low_memory=True** (default) different chunks may infer different data types, causing **Dtypewarning**.

Using **low_memory= False** makes pandas inspect the file more carefully before assigning the data types.

### Step 2.2: Data Inspection

Initial Dataset Inspection

The dataset is inspected to understand its structure, number of records, feature types, and potential issues such as missing values or incorrect data types. This helps in planning appropriate preprocessing steps.

In [6]:
print(df1.shape)
print(df1.info())

(700001, 49)
<class 'pandas.DataFrame'>
RangeIndex: 700001 entries, 0 to 700000
Data columns (total 49 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   0       700001 non-null  str    
 1   1       700001 non-null  str    
 2   2       700001 non-null  str    
 3   3       700001 non-null  str    
 4   4       700001 non-null  str    
 5   5       700001 non-null  str    
 6   6       700001 non-null  float64
 7   7       700001 non-null  int64  
 8   8       700001 non-null  int64  
 9   9       700001 non-null  int64  
 10  10      700001 non-null  int64  
 11  11      700001 non-null  int64  
 12  12      700001 non-null  int64  
 13  13      700001 non-null  str    
 14  14      700001 non-null  float64
 15  15      700001 non-null  float64
 16  16      700001 non-null  int64  
 17  17      700001 non-null  int64  
 18  18      700001 non-null  int64  
 19  19      700001 non-null  int64  
 20  20      700001 non-null  int64  
 21  21      

In [7]:
df1.head()

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


### Step 2.3: Load the Features File

The data files usually don't contain headers. Without headers we can't understand the columns. This features file contains the names and the information about what each column in our dataset tells. So we will use those names as column names for the datasets we have.

#### Checking the Type of Encoding before loading the Features File

Before loading the features file we need to check the type of encoding in which the features file is encoded.

As I worked on this features file earlier, this throws a **UnicodeDecodeError** while loading the file.

Because by default pandas uses UTf-8 as encoding assuming the file was encoded in UTF-8, but the file actually contains bytes that are not valid UTF-8 characters.

We can detect what type of encoding is used in file automatically using a module called **chardet**

Install this module using **pip install chardet**

Afetr installing import the **chardet** module

In [8]:
import chardet

with open(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\NUSW-NB15_features.csv", 'rb') as f:
    result = chardet.detect(f.read())

print(result)

{'encoding': 'ISO-8859-1', 'confidence': 0.2871425945459523, 'language': 'en', 'mime_type': 'text/plain'}


Here the file is probably encoded using **ISO-8859-1** which is also known as **latin-1** in pandas/python.

This encoding is very similar to **CP1251** and **windows-1252**

commonly used in:
- old csv files
- windows exports
- legacy datasets

**confidence: 0.2871425945459523**

This means 28.7% confidence which is ver low confidence. Meaning chardet is not very sure in predicting the encoding technique.

Low confidence because:
- file is probably small
- mostly contains english text
- many encodings looks similar for englisg characters

Here, we can use **latin-1** as encoding because **ISO-8859-1** is **latin-1** in pandas/python.

In [9]:
features = pd.read_csv(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\NUSW-NB15_features.csv",
                      encoding = 'latin-1')

print(features.columns)
print(features.shape)
print(features.head())

Index(['No.', 'Name', 'Type ', 'Description'], dtype='str')
(49, 4)
   No.    Name    Type               Description
0    1   srcip  nominal        Source IP address
1    2   sport  integer       Source port number
2    3   dstip  nominal   Destination IP address
3    4  dsport  integer  Destination port number
4    5   proto  nominal     Transaction protocol


### Step 2.4: Extracting the Feature Names

In [10]:
column_names = features['Name'].tolist()

print(len(column_names))
print(column_names)

49
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


### Step 2.5: Assigning the column Names to dataset as headers

As we already loaded the one of the dataset we need to check one thing

In [12]:
print(len(column_names))
print(df1.shape[1])

49
49


This tells us:
- The feature file contains 49 columns.
- The dataset contains 49 columns.
- The feature files can be safely assigned as headers.

If dataset has less columns compared feature file columns then we should not assign them and it should be handled differntly.

In [13]:
df1.columns = column_names
df1.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


#### Check for Duplicate Columns

In [15]:
df1.columns.duplicated().sum()

np.int64(0)

### Step 2.6: Loading and Data Inspecting the Remaining 3 UNSW-NB15 datasets

In [16]:
df2 = pd.read_csv(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\UNSW-NB15_2.csv",
                 header = None, low_memory = False)

In [17]:
print(df2.shape)
df2.head()

(700001, 49)


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.0,6055,149.171.126.5,54145,tcp,FIN,0.072974,4238,60788,31,...,0,13,13,6,7,1,1,2,NaN,0
1,59.166.0.0,7832,149.171.126.3,5607,tcp,FIN,0.144951,5174,91072,31,...,0,13,13,6,7,1,1,2,NaN,0
2,59.166.0.8,11397,149.171.126.6,21,tcp,FIN,0.116107,2934,3742,31,...,1,1,2,7,5,1,1,4,NaN,0
3,59.166.0.0,3804,149.171.126.3,53,udp,CON,0.000986,146,178,31,...,0,13,13,6,7,1,1,2,NaN,0
4,59.166.0.8,14339,149.171.126.6,14724,tcp,FIN,0.038480,8928,320,31,...,0,8,20,7,5,1,1,4,NaN,0


In [18]:
df3 = pd.read_csv(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\UNSW-NB15_3.csv",
                 header = None, low_memory = False)

In [19]:
print(df3.shape)
df3.head()

(700001, 49)


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.1,18247,149.171.126.4,7662,tcp,FIN,0.119596,4550,68342,31,...,,6,2,2,5,1,1,2,NaN,0
1,59.166.0.3,54771,149.171.126.2,27709,tcp,FIN,0.650574,8928,320,31,...,,3,5,2,4,1,1,4,NaN,0
2,59.166.0.8,13289,149.171.126.9,5190,tcp,FIN,0.007980,2158,2464,31,...,,3,5,1,1,1,1,3,NaN,0
3,149.171.126.18,1043,175.45.176.3,53,udp,INT,0.000005,264,0,60,...,,19,19,19,19,19,19,19,NaN,0
4,149.171.126.18,1043,175.45.176.3,53,udp,INT,0.000005,264,0,60,...,,19,19,19,19,19,19,19,NaN,0


In [20]:
df4 = pd.read_csv(r"C:\Users\yaswa\OneDrive\Desktop\ML_Project_1\Network_Intrusion_Dataset\UNSW-NB15_4.csv",
                 header = None, low_memory = False)

In [21]:
print(df4.shape)
df4.head()

(440044, 49)


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.9,7045,149.171.126.7,25,tcp,FIN,0.201886,37552,3380,31,...,,2,2,7,4,1,1,3,NaN,0
1,59.166.0.9,9685,149.171.126.2,80,tcp,FIN,5.864748,19410,1087890,31,...,,3,1,4,4,1,1,1,NaN,0
2,59.166.0.2,1421,149.171.126.4,53,udp,CON,0.001391,146,178,31,...,,3,5,2,7,1,1,4,NaN,0
3,59.166.0.2,21553,149.171.126.2,25,tcp,FIN,0.053948,37812,3380,31,...,,1,1,4,7,1,1,3,NaN,0
4,59.166.0.8,45212,149.171.126.4,53,udp,CON,0.000953,146,178,31,...,,2,5,2,1,1,1,2,NaN,0


### Step 2.7: Assigning column names to remaining 3 DFs as Headers

In [26]:
df2.columns = column_names
df3.columns = column_names
df4.columns = column_names

In [27]:
df2.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,6055,149.171.126.5,54145,tcp,FIN,0.072974,4238,60788,31,...,0,13,13,6,7,1,1,2,NaN,0
1,59.166.0.0,7832,149.171.126.3,5607,tcp,FIN,0.144951,5174,91072,31,...,0,13,13,6,7,1,1,2,NaN,0
2,59.166.0.8,11397,149.171.126.6,21,tcp,FIN,0.116107,2934,3742,31,...,1,1,2,7,5,1,1,4,NaN,0
3,59.166.0.0,3804,149.171.126.3,53,udp,CON,0.000986,146,178,31,...,0,13,13,6,7,1,1,2,NaN,0
4,59.166.0.8,14339,149.171.126.6,14724,tcp,FIN,0.038480,8928,320,31,...,0,8,20,7,5,1,1,4,NaN,0


In [28]:
df3.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.1,18247,149.171.126.4,7662,tcp,FIN,0.119596,4550,68342,31,...,,6,2,2,5,1,1,2,NaN,0
1,59.166.0.3,54771,149.171.126.2,27709,tcp,FIN,0.650574,8928,320,31,...,,3,5,2,4,1,1,4,NaN,0
2,59.166.0.8,13289,149.171.126.9,5190,tcp,FIN,0.007980,2158,2464,31,...,,3,5,1,1,1,1,3,NaN,0
3,149.171.126.18,1043,175.45.176.3,53,udp,INT,0.000005,264,0,60,...,,19,19,19,19,19,19,19,NaN,0
4,149.171.126.18,1043,175.45.176.3,53,udp,INT,0.000005,264,0,60,...,,19,19,19,19,19,19,19,NaN,0


In [29]:
df4.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.9,7045,149.171.126.7,25,tcp,FIN,0.201886,37552,3380,31,...,,2,2,7,4,1,1,3,NaN,0
1,59.166.0.9,9685,149.171.126.2,80,tcp,FIN,5.864748,19410,1087890,31,...,,3,1,4,4,1,1,1,NaN,0
2,59.166.0.2,1421,149.171.126.4,53,udp,CON,0.001391,146,178,31,...,,3,5,2,7,1,1,4,NaN,0
3,59.166.0.2,21553,149.171.126.2,25,tcp,FIN,0.053948,37812,3380,31,...,,1,1,4,7,1,1,3,NaN,0
4,59.166.0.8,45212,149.171.126.4,53,udp,CON,0.000953,146,178,31,...,,2,5,2,1,1,1,2,NaN,0


### Check for Duplicates

In [30]:
print(df1.columns.duplicated().sum())
print(df2.columns.duplicated().sum())
print(df3.columns.duplicated().sum())
print(df4.columns.duplicated().sum())

0
0
0
0


### Step 2.8: Combine all 4 DFs into single file and taking the Sample data of 100k Records

Why Combining All 4 Files is Better?

The UNSW-NB15 dataset was originally split into multiple CSV files mainly because of size limitations.

The files are usually:

- different partitions of the SAME dataset,
- not different datasets.

If we use only:

UNSW_NB15_1.csv

then we may accidentally:

- miss some attack patterns,
- miss some class distributions,
- miss some feature distributions,
- bias our model.

Which leads to:

- model accuracy becomes misleading.
- real-world performance becomes poor.

In [31]:
df = pd.concat([df1, df2, df3, df4], ignore_index = True)

Why **ignore_index=True** ?

without it: 
- duplicate indexes remain same

with it:
- pandas creates fresh and continuous indexes which would be much cleaner

### Verify Merged Data and Column Names

In [32]:
print(df.shape)

(2540047, 49)


In [33]:
print(df.columns)

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload',
       'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime',
       'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat',
       'Label'],
      dtype='str')


In [34]:
df.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             str    
 1   sport             object 
 2   dstip             str    
 3   dsport            str    
 4   proto             str    
 5   state             str    
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   sttl              int64  
 10  dttl              int64  
 11  sloss             int64  
 12  dloss             int64  
 13  service           str    
 14  Sload             float64
 15  Dload             float64
 16  Spkts             int64  
 17  Dpkts             int64  
 18  swin              int64  
 19  dwin              int64  
 20  stcpb             int64  
 21  dtcpb             int64  
 22  smeansz           int64  
 23  dmeansz           int64  
 24  trans_depth       int64  
 25  res_bdy_len       int64  
 26  Sjit              float64

### Checking NUll values

In [36]:
df.isnull().sum()

srcip                     0
sport                     0
dstip                     0
dsport                    0
proto                     0
state                     0
dur                       0
sbytes                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
Sload                     0
Dload                     0
Spkts                     0
Dpkts                     0
swin                      0
dwin                      0
stcpb                     0
dtcpb                     0
smeansz                   0
dmeansz                   0
trans_depth               0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
Sintpkt                   0
Dintpkt                   0
tcprtt                    0
synack                    0
ackdat                    0
is_sm_ips_ports     

### Calculating the Null values percentage

This tells us the Null values percentage of each column

In [37]:
null_perc = (df.isnull().sum() / len(df))*100
print(null_perc[null_perc>0])

ct_flw_http_mthd    53.075593
is_ftp_login        56.293407
attack_cat          87.351297
dtype: float64


Let's Understand the Null Values

attack_cat = 87.35% Null

This looks alarming at first, but for UNSW-NB15 dataset this is often expected.

Typically:

Label -> 0(Normal) then attack_cat -> NaN
Label -> 1(Attack) then attack_cat -> Exploits
Label -> 1(Attack) then attack_cat -> Fuzzers

And so on.

In many versions of the datasets:
- Normal traffic has no attack category
- Only attack records have attack category

We will confirm this with one more step by using GroupBy:

In [42]:
print(df.groupby('Label')['attack_cat']
     .apply(lambda x: x.isnull().mean()*100))

Label
0    100.0
1      0.0
Name: attack_cat, dtype: float64


This tell us that the missing values are completely from Normal traffic.

For every Normal Network connection: label is 0, the attack_cat value is missing.

For every attack_cat connection: label is 1, the attack_cat is avaliable.

Is this a Data Quality Problem?

NO.

This is expected dataset behavior.

Think about it:

If traffic is normal:

label = 0

then there is no attack category to assign.

Asking:

"What attack category is this normal traffic?"

doesn't make sense.

So the dataset stores:

attack_cat = NaN, for normal traffic.

Here, we don't treat null values for **attack_cat** column because our target variabel is **label** column which is binary classification.

### Duplicate Record Analysis and Removal

To identify and remove duplicate records from the UNSW-NB15 dataset before performing sampling, preprocessing, and model training.

Duplicate records can introduce bias into machine learning models because the same network traffic instances may appear multiple times in the dataset.

This can lead to:
- Over-representation of certain patterns
- Biased model learning
- Inflated performance metrics
- Increased computational cost during training

Therefore, duplicate detection and removal were performed as an important data cleaning step.

### Duplicate Detection

The Pandas function duplicated() was used to identify records that appeared more than once in the dataset.

In [38]:
df.duplicated().sum()

np.int64(480632)

This indicates that 480,632 rows were exact duplicates of previously occurring records.

### Calculating duplicates percentage

To understand the extent of duplication, the percentage of duplicate records was calculated using:

In [41]:
duplicate_perc = df.duplicated().mean()*100
print(f"{duplicate_perc:.2f}%")

18.92%


**Interpretation**

Approximately 18.92% of the dataset consisted of duplicate records.

This is a significant proportion and could negatively affect model training if retained.

### Duplicate Removal

In [43]:
df = df.drop_duplicates()

In [44]:
print(df.shape)
print(df.duplicated().sum())

(2059415, 49)
0


**Impact on Machine Learning Model**

Removing duplicate records provides several benefits:

- Prevents the model from memorizing repeated instances.
- Reduces bias caused by overrepresented network traffic patterns.
- Improves the quality and diversity of training data.
- Reduces memory usage and training time.
- Produces more reliable evaluation metrics.

Therefore, duplicate removal was performed as a mandatory data preprocessing step before proceeding with sampling, feature engineering, and K-Nearest Neighbors (KNN) model development.

### Target Variable Distribution Analysis

Before performing stratified sampling, the distribution of the target variable (Label) was analyzed to understand the proportion of normal and attack traffic records present in the UNSW-NB15 dataset.

The target variable is defined as:

- Label = 0 → Normal Traffic
- Label = 1 → Attack Traffic

**Class Distribution Analysis**

The frequency of each class was obtained using:

In [45]:
print(df['Label'].value_counts())

Label
0    1959772
1      99643
Name: count, dtype: int64


To determine the percentage distribution:

In [46]:
print(df['Label'].value_counts(normalize = True)*100)

Label
0    95.161587
1     4.838413
Name: proportion, dtype: float64


The dataset is highly imbalanced, with:

- 95.16% Normal Traffic
- 4.84% Attack Traffic

This indicates that normal network traffic significantly outnumbers attack traffic instances.

**Why This Step Was Performed?**

The class distribution was examined before sampling to determine the proportion of each class in the original dataset.

Since the dataset is imbalanced, stratified sampling was later used to ensure that the sampled dataset maintains the same class proportions as the original dataset.

Without stratification, random sampling could result in:

- Underrepresentation of attack records
- Changes in class distribution
- Biased model training and evaluation

Therefore, analyzing the target variable distribution was necessary to guide the sampling strategy and preserve the characteristics of the original dataset.

**Conclusion**

The original UNSW-NB15 dataset contains approximately 95.16% normal traffic and 4.84% attack traffic. These proportions were used as the basis for stratified sampling to ensure that the sampled dataset remained representative of the original data.

## Creating a Stratified Sample Dataset

The cleaned UNSW-NB15 dataset contains over 2 million network traffic records. Training and evaluating machine learning algorithms such as K-Nearest Neighbors (KNN) on the entire dataset can be computationally expensive and time-consuming.

Therefore, a representative sample of 100,000 records was created to reduce computational complexity while preserving the original characteristics of the dataset.

In [48]:
from sklearn.model_selection import train_test_split

sample_df, reamining_df = train_test_split(df, train_size = 100000, stratify = df['Label'], random_state = 42)



**Explanation of Parameters**
- sample_df: It will store our 100k records.
- remaining_df: It will store the reamining records which we dont use.
- df: Original cleaned dataset.
- train_size=100000: Selects 100,000 records for the sample dataset.
- stratify=df['Label']: Preserves the original class distribution in the sample.
- random_state=42: Ensures reproducibility, meaning the same sample will be generated every time the code is executed.

In [49]:
print(sample_df.shape)
sample_df.head()

(100000, 49)


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
1544416,59.166.0.8,17724,149.171.126.5,18149,tcp,FIN,0.071061,8928,320,31,...,,6,3,2,4,1,1,5,NaN,0
418490,59.166.0.4,25046,149.171.126.6,21,tcp,FIN,0.767884,2934,3738,31,...,4,6,6,18,23,6,6,14,NaN,0
627965,59.166.0.7,56853,149.171.126.9,143,tcp,FIN,0.038289,7814,15116,31,...,0,8,4,2,2,1,1,1,NaN,0
1090737,59.166.0.5,18464,149.171.126.3,6881,tcp,FIN,0.423522,2302,34406,31,...,,17,9,10,9,6,1,9,NaN,0
2082402,59.166.0.3,38437,149.171.126.6,25409,tcp,FIN,0.235664,8928,320,31,...,,2,2,6,3,1,1,4,NaN,0


**Result**

A stratified sample dataset containing 100,000 records was successfully created. The sample preserves the original distribution of normal and attack traffic and serves as a computationally efficient subset for subsequent preprocessing, feature engineering, and KNN model training.

### Verifying sample distribution

After creating the stratified sample dataset, the class distribution was verified to ensure that the sampling process preserved the original proportions of normal and attack traffic records.

This validation step confirms that the sampled dataset remains representative of the original UNSW-NB15 dataset.

**Class Distribution Verification**

The percentage distribution of the target variable (Label) was calculated using:

In [50]:
print(sample_df['Label'].value_counts(normalize = True)*100)

Label
0    95.162
1     4.838
Name: proportion, dtype: float64


These proportions are nearly identical to those observed in the original dataset.

This confirms that the stratified sampling process successfully preserved the original class distribution.

**Conclusion**

The sampled dataset accurately represents the original UNSW-NB15 dataset in terms of target class proportions and is therefore suitable for further preprocessing and machine learning model development.

### Resetting the Index

After sampling, the selected records retain their original row indices from the full dataset. As a result, the index values may become non-sequential.

To improve dataset organization and maintain a clean structure, the index was reset.

In [51]:
sample_df = sample_df.reset_index(drop = True)

**Explanation**

- reset_index(): creates a new sequential index.
- drop=True: removes the old index instead of adding it as a separate column.

### Saving the Sample Dataset

The finalized sampled dataset was saved as a CSV file for future use in preprocessing, feature engineering, model training, and evaluation.

In [52]:
sample_df.to_csv("UNSW-NB15_100K.csv", index = False)

**Explanation**

- to_csv(): exports the DataFrame to a CSV file.
- index=False: prevents the DataFrame index from being stored as an additional column.

**Result**

A stratified sample dataset containing 100,000 records was successfully saved as:

UNSW-NB15_100K.csv

This file serves as the working dataset for the subsequent stages of the Network Intrusion Detection System (NIDS) project.

## Loading and Verifying the Stratified Sample Dataset

**Objective**

After creating and saving the stratified sample dataset, the dataset was reloaded into the notebook for further preprocessing, exploratory data analysis, feature engineering, and machine learning model development.

Using the sampled dataset significantly reduces computational requirements while maintaining the original characteristics of the UNSW-NB15 dataset.

**Dataset Loading**

The sampled dataset was loaded using the Pandas read_csv() function.

In [126]:
samp_df = pd.read_csv("UNSW-NB15_100K.csv", low_memory = False)

**Explanation**

- pd.read_csv(): Reads the CSV file into a Pandas DataFrame.
- UNSW-NB15_100K.csv: The previously created stratified sample dataset containing 100,000 records.
- low_memory=False: Ensures that Pandas reads the dataset in a single pass, allowing more accurate data type inference and avoiding potential DtypeWarning messages.

**Dataset Shape Verification**

The shape of the dataset was checked to confirm that the file was loaded successfully.

In [127]:
print(samp_df.shape)

(100000, 49)


**Previewing the Dataset**

To ensure the dataset had benn loaded perfectly

In [128]:
samp_df.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.8,17724,149.171.126.5,18149,tcp,FIN,0.071061,8928,320,31,...,,6,3,2,4,1,1,5,NaN,0
1,59.166.0.4,25046,149.171.126.6,21,tcp,FIN,0.767884,2934,3738,31,...,4,6,6,18,23,6,6,14,NaN,0
2,59.166.0.7,56853,149.171.126.9,143,tcp,FIN,0.038289,7814,15116,31,...,0,8,4,2,2,1,1,1,NaN,0
3,59.166.0.5,18464,149.171.126.3,6881,tcp,FIN,0.423522,2302,34406,31,...,,17,9,10,9,6,1,9,NaN,0
4,59.166.0.3,38437,149.171.126.6,25409,tcp,FIN,0.235664,8928,320,31,...,,2,2,6,3,1,1,4,NaN,0


In [129]:
samp_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 49 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   srcip             100000 non-null  str    
 1   sport             100000 non-null  str    
 2   dstip             100000 non-null  str    
 3   dsport            100000 non-null  str    
 4   proto             100000 non-null  str    
 5   state             100000 non-null  str    
 6   dur               100000 non-null  float64
 7   sbytes            100000 non-null  int64  
 8   dbytes            100000 non-null  int64  
 9   sttl              100000 non-null  int64  
 10  dttl              100000 non-null  int64  
 11  sloss             100000 non-null  int64  
 12  dloss             100000 non-null  int64  
 13  service           100000 non-null  str    
 14  Sload             100000 non-null  float64
 15  Dload             100000 non-null  float64
 16  Spkts             100000 non-nul

## Checking Duplicates

After loading the stratified sample dataset, a duplicate record check was performed to verify that the dataset contained only unique observations.

Since duplicate records had already been removed during the data cleaning stage, this step was carried out as a validation measure before proceeding with further preprocessing and machine learning model development.

In [130]:
samp_df.duplicated().sum()

np.int64(0)

**Interpretation**

The result indicates that the sampled dataset contains zero duplicate records.

**Conclusion**

The duplicate verification process confirmed that the sampled UNSW-NB15 dataset contains 100,000 unique records with no duplicate observations, making it suitable for further analysis and machine learning tasks.

## Checking Missing Values

In [131]:
samp_df.isnull().sum()

srcip                   0
sport                   0
dstip                   0
dsport                  0
proto                   0
state                   0
dur                     0
sbytes                  0
dbytes                  0
sttl                    0
dttl                    0
sloss                   0
dloss                   0
service                 0
Sload                   0
Dload                   0
Spkts                   0
Dpkts                   0
swin                    0
dwin                    0
stcpb                   0
dtcpb                   0
smeansz                 0
dmeansz                 0
trans_depth             0
res_bdy_len             0
Sjit                    0
Djit                    0
Stime                   0
Ltime                   0
Sintpkt                 0
Dintpkt                 0
tcprtt                  0
synack                  0
ackdat                  0
is_sm_ips_ports         0
ct_state_ttl            0
ct_flw_http_mthd    45574
is_ftp_login

### Checking Null values Percentage

In [132]:
null_perct = (samp_df.isnull().sum() / len(samp_df))*100
print(null_perct[null_perct>0].sort_values(ascending = False))

attack_cat          95.162
is_ftp_login        49.479
ct_flw_http_mthd    45.574
dtype: float64


 We have already determined:
- **attack_cat** is expected to be null for normal traffic
- **attack_cat** is also not used as feature because our target variable is **Label**

So we will focus on these columns:
- is_ftp_login      
- ct_flw_http_mthd 

## Understanding the Meaning of the Columns

Before handling missing values, it is important to understand the meaning and behavior of the features within the UNSW-NB15 dataset.

Certain network traffic features are only applicable to specific services or protocols. Therefore, apparent missing values may not necessarily indicate incomplete data; instead, they may represent situations where a feature is not relevant to a particular network service.

To validate this assumption, the relationship between the service feature and selected columns was examined.

### Feature Analysis

The following features were investigated:

- **is_ftp_login**: Indicates whether an FTP login attempt occurred.
- **ct_flw_http_mthd**: Represents the count of HTTP methods observed within a flow.

These features are expected to be meaningful only for specific services:

**is_ftp_login** → Primarily associated with FTP traffic.
**ct_flw_http_mthd** → Primarily associated with HTTP traffic.

In [133]:
samp_df.groupby('service')['is_ftp_login'].agg(['count', 'size']).sort_values('size', ascending=False)

,count,size
service,,
-,29234,56655
dns,8561,18859
http,4514,9815
ftp-data,3041,6119
smtp,1903,4005
ssh,1121,2265
ftp,2139,2191
pop3,6,73
dhcp,0,7


In [134]:
samp_df.groupby('service')['ct_flw_http_mthd'].agg(['count', 'size']).sort_values('size', ascending=False)

,count,size
service,,
-,29284,56655
dns,8561,18859
http,9447,9815
ftp-data,3041,6119
smtp,1903,4005
ssh,1121,2265
ftp,1061,2191
pop3,6,73
dhcp,0,7


#### Explanation

In the UNSW-NB15 dataset, **service** generally describes the network service being used.

Think of it as:

Which application/service generated this traffic?

The dataset was grouped by the **service** column, and two statistics were calculated:

- **count**: Number of non-null values.
- **size**: Total number of records in each service category.

Comparing these values helps identify whether missing values occur systematically for particular services.

#### Purpose

This analysis was performed to determine whether missing values represent:

Actual missing information resulting from data collection issues.
Service-dependent values where the feature is naturally unavailable for certain network services.

## Determine whether Missing Means are "Not Applicable"

Missing values are service-dependent rather than random.
- 'is_ftp_login' is only applicable to FTP traffic, while
- 'ct_flw_http_mthd' is only applicable to HTTP traffic.
- Therefore, the missing values represent "not applicable"
- conditions rather than incomplete or corrupted data.

In [135]:
print(samp_df['is_ftp_login'].unique())
print(samp_df['ct_flw_http_mthd'].unique())

[nan  1.  0.  4.  2.]
[nan  0.  1.  6.  4.  3.  2. 14.  8.  5.  9. 30. 16. 12.]


In [136]:
print(samp_df['is_ftp_login'].value_counts(dropna=False))

is_ftp_login
NaN    49479
0.0    48563
1.0     1950
4.0        6
2.0        2
Name: count, dtype: int64


In [137]:
print(samp_df['ct_flw_http_mthd'].value_counts(dropna=False))

ct_flw_http_mthd
NaN     45574
0.0     44795
1.0      9091
6.0       255
4.0       205
3.0        28
2.0        21
9.0        11
5.0        10
14.0        3
8.0         3
12.0        2
30.0        1
16.0        1
Name: count, dtype: int64


The features is_ftp_login and ct_flw_http_mthd contained approximately 50% missing values. Analysis of the value distributions showed that the majority of non-missing observations were already equal to 0, indicating that missing values likely represented absence of FTP login activity or HTTP method activity rather than random data loss. Therefore, missing values can be replaced with 0.

## Treating Missing values

In [138]:
samp_df['is_ftp_login'] = samp_df['is_ftp_login'].fillna(0)
samp_df['ct_flw_http_mthd'] = samp_df['ct_flw_http_mthd'].fillna(0)

**Why Is 0 Reasonable Here?**

For FTP feature:

No FTP login observed

can reasonably be represented as: 0

For HTTP methods:

No HTTP methods observed

can reasonably be represented as: 0

**Why Not Use Mean/Median?**

This feature is essentially binary/indicator-like.

Example:

is_ftp_login = 0.42

If filled with Mean/Meadian.

makes no logical sense.

## Verify After Filling Null Values

In [139]:
print(samp_df[['is_ftp_login', 'ct_flw_http_mthd']].isnull().sum())

is_ftp_login        0
ct_flw_http_mthd    0
dtype: int64


## What about attack_cat Column?

Since our target variable is Label column which is Binary Classification, we need to drop the attack_cat column. As this column contains different types of attacks, dropping this column avoids target leakage while training the Model

In [140]:
samp_df = samp_df.drop(columns = ['attack_cat'])

In [141]:
print('attack_cat' in samp_df.columns)

False


# Split the Data into Inputs(X) and Output(y)

Before training the machine learning model, the dataset is seperated into input features (X) and traget variable (y).

In supervised machine learning, the model learns the relationship between the input features and the target variable. Therefore, the target column must be isolated from the predictor variables before proceeding with model training.

**Target Variable**

The label column was selected as target variable.

The target label represents:
- 0 -> Normal Netwrok Traffic
- 1 -> Attack Traffic

The machine learning model will use the input features to predict whether a network traffic record corresponds to normal behavior or a network intrusion.

In [142]:
X = samp_df.drop(columns = ['Label'])
y = samp_df[['Label']]

After the split:

- X contains all network traffic features used for intrusion detection.
- y contains the corresponding class labels indicating normal or attack traffic.

These datasets will be used in the subsequent train-test split, preprocessing, feature transformation, and K-Nearest Neighbors (KNN) model training stages.

In [143]:
X.head()

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
0,59.166.0.8,17724,149.171.126.5,18149,tcp,FIN,0.071061,8928,320,31,...,0.0,0.0,,6,3,2,4,1,1,5
1,59.166.0.4,25046,149.171.126.6,21,tcp,FIN,0.767884,2934,3738,31,...,0.0,1.0,4,6,6,18,23,6,6,14
2,59.166.0.7,56853,149.171.126.9,143,tcp,FIN,0.038289,7814,15116,31,...,0.0,0.0,0,8,4,2,2,1,1,1
3,59.166.0.5,18464,149.171.126.3,6881,tcp,FIN,0.423522,2302,34406,31,...,0.0,0.0,,17,9,10,9,6,1,9
4,59.166.0.3,38437,149.171.126.6,25409,tcp,FIN,0.235664,8928,320,31,...,0.0,0.0,,2,2,6,3,1,1,4


In [144]:
y.head()

,Label
0,0
1,0
2,0
3,0
4,0


## Split the Data into Train and Test Set (80-20 Rule)

After separating the features (X) and target variable (y), the dataset was divided into training and testing sets.

The purpose of this split is to train the machine learning model on one subset of the data and evaluate its performance on previously unseen data. This helps measure the model's ability to generalize to new network traffic records.|

**Why Train-Test Splitting?**

Machine learning models should not be evaluated using the same data on which they were trained.

If the model is tested on the training data, the performance metrics may be overly optimistic and may not reflect real-world performance.

Therefore, the dataset was divided into:

- Training Set (80%) → Used to train the KNN model.
- Testing Set (20%) → Used to evaluate the trained model.

In [145]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, stratify = y, random_state = 44)

The dataset was successfully divided into training and testing sets using stratified sampling. This approach preserved the original class distribution of the UNSW-NB15 dataset and provided representative datasets for K-Nearest Neighbors (KNN) model training and evaluation.

In [146]:
X_train

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
87547,59.166.0.9,1177,149.171.126.3,31619,tcp,FIN,0.023626,3286,38766,31,...,0.0,0.0,0,2,3,1,2,1,1,1
96068,175.45.176.2,39712,149.171.126.15,80,tcp,FIN,0.462613,566,268,254,...,1.0,0.0,,3,3,2,2,2,1,3
87217,59.166.0.7,41499,149.171.126.2,143,tcp,FIN,0.031701,7814,15688,31,...,0.0,0.0,0,5,6,1,1,1,1,1
70108,59.166.0.7,55625,149.171.126.7,11513,tcp,FIN,0.078129,320,1818,31,...,0.0,0.0,0,1,1,2,2,1,1,1
64385,59.166.0.0,16027,149.171.126.0,35429,tcp,FIN,0.128398,4238,65524,31,...,0.0,0.0,0,13,4,8,4,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83457,59.166.0.1,44097,149.171.126.4,49160,tcp,FIN,0.058949,8928,320,31,...,0.0,0.0,,1,1,5,2,1,1,2
84069,59.166.0.7,26154,149.171.126.3,25,tcp,FIN,0.020701,37172,3276,31,...,0.0,0.0,,1,1,1,2,1,1,1
23141,59.166.0.4,46733,149.171.126.7,5190,tcp,FIN,3.613176,13360,45624,31,...,0.0,0.0,,7,4,1,3,1,1,2
34326,59.166.0.0,40077,149.171.126.8,10232,tcp,FIN,0.023799,2646,25956,31,...,0.0,0.0,0,6,5,4,5,1,1,1


In [147]:
X_test

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
28555,59.166.0.7,43473,149.171.126.8,53,udp,CON,0.001070,146,178,31,...,0.0,0.0,0,4,4,2,3,2,1,1
77375,59.166.0.2,57013,149.171.126.1,22,tcp,FIN,0.006124,3728,5474,31,...,0.0,0.0,,2,3,4,4,1,1,2
5615,59.166.0.2,62718,149.171.126.1,80,tcp,FIN,1.238820,1684,10168,31,...,0.0,0.0,,1,1,2,3,1,1,2
99292,59.166.0.5,53803,149.171.126.8,6881,tcp,FIN,0.026142,1540,1644,31,...,0.0,0.0,,12,13,4,7,4,1,5
78484,59.166.0.0,42372,149.171.126.2,4634,tcp,FIN,0.236218,2662,23470,31,...,0.0,0.0,,3,2,1,2,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67074,175.45.176.2,3540,149.171.126.13,520,udp,INT,0.000001,104,0,254,...,0.0,0.0,,2,7,3,1,1,1,3
81043,59.166.0.4,49828,149.171.126.7,63666,tcp,FIN,0.222391,8928,320,31,...,0.0,0.0,,3,3,4,7,1,1,4
2492,59.166.0.0,9865,149.171.126.7,80,tcp,FIN,1.005652,1580,10168,31,...,1.0,0.0,,2,2,1,5,2,1,1
7746,59.166.0.1,47137,149.171.126.6,30385,tcp,FIN,0.009776,2230,13482,31,...,0.0,0.0,0,4,6,1,4,1,1,1


In [148]:
y_train

,Label
87547,0
96068,1
87217,0
70108,0
64385,0
...,...
83457,0
84069,0
23141,0
34326,0


In [149]:
y_test

,Label
28555,0
77375,0
5615,0
99292,0
78484,0
...,...
67074,1
81043,0
2492,0
7746,0


## Categorizing Features Based on Preprocessing Requirements

After splitting the dataset into training and testing sets, the input features were categorized into separate groups according to the type of preprocessing required.

Different machine learning features often require different transformation techniques depending on their data type and characteristics. Therefore, organizing features into preprocessing groups simplifies the data preparation pipeline and ensures that each feature receives the appropriate transformation.

### Feature Categories

The training dataset is divided into 3 groups:


**1.Categorical Features**

This group contains categorical variables represented as text or discrete categories.

Since machine learning algorithms cannot directly process textual categories, these features require encoding techniques such as:

- One-Hot Encoding
- Ordinal Encoding
- Frequency Encoding

depending on the feature characteristics.

In [150]:
cat_df = X_train.select_dtypes(include = ['object', 'str'])
cat_df

,srcip,sport,dstip,dsport,proto,state,service,ct_ftp_cmd
87547,59.166.0.9,1177,149.171.126.3,31619,tcp,FIN,-,0
96068,175.45.176.2,39712,149.171.126.15,80,tcp,FIN,http,
87217,59.166.0.7,41499,149.171.126.2,143,tcp,FIN,-,0
70108,59.166.0.7,55625,149.171.126.7,11513,tcp,FIN,ftp-data,0
64385,59.166.0.0,16027,149.171.126.0,35429,tcp,FIN,-,0
...,...,...,...,...,...,...,...,...
83457,59.166.0.1,44097,149.171.126.4,49160,tcp,FIN,ftp-data,
84069,59.166.0.7,26154,149.171.126.3,25,tcp,FIN,smtp,
23141,59.166.0.4,46733,149.171.126.7,5190,tcp,FIN,-,
34326,59.166.0.0,40077,149.171.126.8,10232,tcp,FIN,-,0


**Checking whether all categorical columns are actual string values or numerical values are stored as string which turn them into categorical.**

In [151]:
cat_df['sport'].unique()

<StringArray>
[ '1177', '39712', '41499', '55625', '16027', '36346',  '1493',  '9041',
 '44031',  '7902',
 ...
 '49224', '14601', '19793',  '4168', '52195', '39142', '22634', '17929',
 '50388', '26154']
Length: 44809, dtype: str

In [152]:
cat_df['dsport'].unique()

<StringArray>
['31619',    '80',   '143', '11513', '35429', '14955', '18093',    '53',
 '59699',    '25',
 ...
 '29210', '45292', '37570', '17233', '35206', '39107', '56491',  '2466',
 '10232',  '1335']
Length: 21879, dtype: str

In [153]:
cat_df['ct_ftp_cmd'].unique()

<StringArray>
['0', ' ', '1', '2', '4', '3', '6', '5']
Length: 8, dtype: str

After extracting the categorical features, a detailed inspection of the columns was performed to verify whether all selected features were truly categorical.

During this analysis, it was observed that some columns contained numerical values but were stored using the **object** data type. As a result, these features were incorrectly classified as categorical variables.

## Data Type Correction: Converting Numerically Encoded Features Stored as Strings

During feature inspection, it was observed that the **sport** (source port) and **dsport** (destination port) columns were stored as object data types. Although these columns primarily contain port numbers, some records contained non-numeric values represented as strings.

Before converting these columns to numerical data types, it was necessary to handle these non-numeric entries to ensure successful conversion and maintain data consistency.

In [154]:
for col in ['sport', 'dsport']:
    check = samp_df[col].astype(str).str.contains('0x', na=False)

    print(f"\n{col}")
    print("hex count:", check.sum())

    if mask.sum() > 0:
        print(samp_df.loc[check, col].unique())


sport
hex count: 1
<StringArray>
['0x000c']
Length: 1, dtype: str

dsport
hex count: 12
<StringArray>
['0xcc09', '0xc0a8']
Length: 2, dtype: str


The majority of values in the **sport** and **dsport** columns represented numerical port numbers. However, a small number of records contained string-based values that prevented direct conversion to numeric data types.

As a result, these columns were initially classified as categorical features despite representing numerical information.

so the 'sport' and 'dsport' columns contains hexa decimal values we cannot convert those columns directly from 'str' to 'int'.

First, we need to convert hexdecimal values to decimal values and then convert all values into 'int'.

In [155]:
print((X_train['dsport'] == '-').sum())

1


During exploratory analysis, it was discovered that the dsport column contained records with the value '-'.

The invalid placeholder values ('-') were identified and handled appropriately before converting the dsport column to a numerical datatype.

The '-' symbol does not represent a valid destination port number and cannot be directly converted to a numerical data type.

#### Preprocessing Strategy

A custom function named convert_port() was created to standardize the port values.

The function performs the following operations:

1. Handle Invalid Values:

Records containing the placeholder value '-' were converted to NaN.

This allows missing values to be handled appropriately during subsequent preprocessing.

2. Convert Hexadecimal Values:

Port values beginning with the prefix 0x were identified as hexadecimal numbers and converted to their decimal equivalents.

3. Convert Standard Port Numbers:

All remaining values were converted directly to integers.


In [156]:
def convert_port(x):
    x = str(x).strip()

    if x == '-':
        return np.nan

    if x.startswith('0x'):
        return int(x, 16)

    return int(x)

The conversion function was applied to both source and destination port features:

In [157]:
X_train['sport'] = X_train['sport'].apply(convert_port)
X_test['sport'] = X_test['sport'].apply(convert_port)

In [158]:
X_train['dsport'] = X_train['dsport'].apply(convert_port)
X_test['dsport'] = X_test['dsport'].apply(convert_port)

To prevent data leakage, the same transformation logic was applied independently to both the training and testing datasets.

**Conclusion**

The sport and dsport features were successfully cleaned and converted from object data types to numerical values. Invalid placeholders were replaced with missing values, hexadecimal ports were converted to decimal format, and all remaining port numbers were standardized for machine learning analysis.

### DataType Verification

In [159]:
print(X_train['sport'].dtype)
print(X_test['sport'].dtype)

int64
int64


In [160]:
print(X_train['dsport'].dtype)
print(X_test['dsport'].dtype)

float64
float64


The verification process confirmed that the sport and dsport features were successfully converted from object data types to numerical formats. The sport feature contains only valid integer values, while the dsport feature remains numerical and retains missing values for subsequent preprocessing.

In [161]:
print(X_train[['sport', 'dsport']].isnull().sum())
print(X_test[['sport', 'dsport']].isnull().sum())

sport     0
dsport    1
dtype: int64
sport     0
dsport    1
dtype: int64


In [162]:
# Filling those missing value with zero

X_train['dsport'] = X_train['dsport'].fillna(0)
X_test['dsport'] = X_test['dsport'].fillna(0)

In [163]:
print(X_train[['sport', 'dsport']].isnull().sum())
print(X_test[['sport', 'dsport']].isnull().sum())

sport     0
dsport    0
dtype: int64
sport     0
dsport    0
dtype: int64


In [164]:
samp_df['ct_ftp_cmd'].unique()

<StringArray>
[' ', '4', '0', '1', '6', '2', '3', '5']
Length: 8, dtype: str

In [165]:
samp_df['ct_ftp_cmd'].value_counts()

ct_ftp_cmd
     49479
0    48560
1     1915
2       16
3       11
4       10
6        5
5        4
Name: count, dtype: int64

This "ct_ftp_cmd" columnn is also stored as 'string' which contains numeric records.

Also contains empty strings which refers to No FTP Command Activity that is already stored as zero.

So we replace empty string with zero and convert the whole column into 'int' dtype.

In [166]:
X_train['ct_ftp_cmd'] = X_train['ct_ftp_cmd'].replace(' ', 0)
X_test['ct_ftp_cmd'] = X_test['ct_ftp_cmd'].replace(' ', 0)

**Converting to Numeric**

In [167]:
X_train['ct_ftp_cmd'] = pd.to_numeric(X_train['ct_ftp_cmd'])
X_test['ct_ftp_cmd'] = pd.to_numeric(X_test['ct_ftp_cmd'])

**Datatype Verification**

In [168]:
X_train['ct_ftp_cmd'].dtype

dtype('int64')

In [169]:
cat_cols = X_train.select_dtypes(include = 'str').columns
cat_cols

Index(['srcip', 'dstip', 'proto', 'state', 'service'], dtype='str')

### Removing IP Address Features (srcip and dstip)

The srcip (source IP address) and dstip (destination IP address) features were examined to determine their suitability for machine learning model training.

These features contain IP addresses represented as string values rather than numerical measurements.

The unique values and frequency distributions of the IP address columns were inspected using:

In [170]:
print(samp_df['srcip'].value_counts())
print(samp_df['dstip'].value_counts())

srcip
59.166.0.0         9420
59.166.0.4         9416
59.166.0.1         9331
59.166.0.5         9326
59.166.0.2         9254
59.166.0.3         9241
59.166.0.8         9024
59.166.0.9         8965
59.166.0.6         8934
59.166.0.7         8744
175.45.176.1       1918
175.45.176.3       1828
175.45.176.0       1672
175.45.176.2       1297
149.171.126.18      432
149.171.126.15      152
149.171.126.14      143
10.40.85.1          141
10.40.182.1         139
149.171.126.12      137
149.171.126.10      116
10.40.182.3          70
10.40.85.30          64
10.40.170.2          59
10.40.85.10          27
149.171.126.4        17
149.171.126.6        16
149.171.126.0        16
149.171.126.5        15
149.171.126.2        14
149.171.126.8        14
149.171.126.1        14
149.171.126.3        12
149.171.126.9        11
149.171.126.7         7
192.168.241.243       6
149.171.126.13        4
10.40.182.6           3
149.171.126.17        1
Name: count, dtype: int64
dstip
149.171.126.4      9442
14

The analysis revealed that both features contain a large number of distinct IP addresses represented as text strings.

These values function primarily as identifiers rather than measurable numerical attributes.

**Why These Features Were Removed**

The K-Nearest Neighbors (KNN) algorithm is a distance-based machine learning algorithm that relies on numerical feature values to calculate similarities between observations.

1. High Cardinality

IP address columns contain a very large number of unique values.

Encoding such features would create an extremely high-dimensional feature space, increasing computational complexity and memory requirements.

2. Lack of Meaningful Distance

Although IP addresses can be converted into numerical representations, the resulting numeric values do not represent meaningful distances between network hosts.

3. Risk of Overfitting

IP addresses often act as identifiers rather than behavioral features.

Including them may cause the model to memorize specific hosts instead of learning generalized patterns associated with normal and malicious network traffic.

4. Increased Computational Cost

KNN stores all training instances and computes distances during prediction. High-cardinality identifier features can unnecessarily increase computational overhead without contributing useful predictive information.

**Preprocessing Decision**

To avoid these issues, the IP address features were removed from both the training and testing datasets.

In [171]:
drop_cols = ['srcip', 'dstip']
X_train = X_train.drop(columns = drop_cols)
X_test = X_test.drop(columns = drop_cols)

The srcip and dstip features were removed because they represent high-cardinality identifier variables that do not provide meaningful distance information for the KNN algorithm. Excluding these features improves model efficiency and supports better generalization during intrusion detection.

In [172]:
cat_cols = X_train.select_dtypes(include = ['str']).columns
cat_cols

Index(['proto', 'state', 'service'], dtype='str')

**Unique Value Analysis**

In [173]:
print("proto :", X_train['proto'].nunique())
print("state :", X_train['state'].nunique())
print("service :", X_train['service'].nunique())

proto : 112
state : 14
service : 12


In [174]:
print(X_train['proto'].value_counts())
print(X_train['state'].value_counts())
print(X_train['service'].value_counts())

proto
tcp         56232
udp         22921
arp           248
unas          162
ospf          160
            ...  
wb-expak        1
xns-idp         1
vines           1
trunk-1         1
ttp             1
Name: count, Length: 112, dtype: int64
state
FIN    55648
CON    21151
INT     2891
REQ      261
RST       20
ECO        9
URH        6
CLO        5
ACC        3
MAS        2
no         1
TXD        1
PAR        1
TST        1
Name: count, dtype: int64
service
-           45447
dns         15042
http         7866
ftp-data     4868
smtp         3191
ssh          1777
ftp          1741
pop3           54
ssl             5
snmp            4
dhcp            4
radius          1
Name: count, dtype: int64


In [175]:
print(X_train['service'].unique())

<StringArray>
[       '-',     'http', 'ftp-data',      'dns',     'smtp',      'ftp',
      'ssh',     'pop3',      'ssl',   'radius',     'snmp',     'dhcp']
Length: 12, dtype: str


This analysis revealed that there is an unknown ('-') category present in **service** column.

Unlike missing values in other features, this value does not represent incomplete data. Instead, it indicates that no specific application-layer service was identified for the corresponding network flow.

Therefore, '-' represents a valid service category rather than a missing observation.

**Preprocessing Decision**

The '-' category was retained without modification.

Replacing it with another service category or treating it as missing data could introduce incorrect information into the dataset and distort the underlying network traffic characteristics.

During categorical encoding, the '-' category will be treated as a separate valid class and represented accordingly in the transformed feature space.

**Conclusion**

The service feature contains a valid category represented by '-', indicating that no specific service was identified. Since this value carries meaningful information and does not represent missing data, it was retained for subsequent encoding and model training.

In [176]:
cat_df = X_train.select_dtypes(include = 'str')
cat_df

,proto,state,service
87547,tcp,FIN,-
96068,tcp,FIN,http
87217,tcp,FIN,-
70108,tcp,FIN,ftp-data
64385,tcp,FIN,-
...,...,...,...
83457,tcp,FIN,ftp-data
84069,tcp,FIN,smtp
23141,tcp,FIN,-
34326,tcp,FIN,-


## Feature Engineering on Categorical Columns

#### One-Hot Encoding of Categorical Features

The categorical features identified in the training dataset were transformed into numerical representations using One-Hot Encoding (OHE).

Machine learning algorithms such as K-Nearest Neighbors (KNN) cannot directly process categorical text values. Therefore, categorical variables must be converted into a numerical format before model training.

**Why One-Hot Encoding?**

The categorical features in the UNSW-NB15 dataset include:

- proto (network protocol)
- service (application service)
- state (connection state)

These features contain nominal categories that do not possess any natural ordering.

**Implementation**

In [177]:
from sklearn.preprocessing import OneHotEncoder

OHE = OneHotEncoder(sparse_output = False, handle_unknown = 'ignore').set_output(transform = 'pandas')

cat_df_trans = OHE.fit_transform(cat_df)

**Parameter Explanation**

- sparse_output=False

Returns the encoded output as a dense matrix instead of a sparse matrix.

This makes the transformed data easier to inspect and combine with other processed features.

- handle_unknown='ignore'

Ensures that previously unseen categories encountered during testing or future prediction do not cause errors.

Unknown categories are automatically ignored during transformation.

- set_output(transform='pandas')

Returns the encoded features as a Pandas DataFrame instead of a NumPy array, preserving column names and improving readability.

**Result**

The categorical features were successfully transformed into numerical binary variables and stored in the DataFrame cat_df_trans.

The transformed dataset can now be combined with the processed numerical features for subsequent preprocessing and KNN model training.

In [178]:
cat_df_trans

,proto_a/n,proto_aes-sp3-d,proto_any,proto_argus,proto_aris,proto_arp,proto_ax.25,proto_bbn-rcc,proto_bna,proto_cbt,...,service_dns,service_ftp,service_ftp-data,service_http,service_pop3,service_radius,service_smtp,service_snmp,service_ssh,service_ssl
87547,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96068,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
87217,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
70108,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
64385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83457,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84069,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
23141,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
34326,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [179]:
num_cols = X_train.select_dtypes(include = ['int', 'float']).columns.tolist()

This **num_cols** stores all the numerical columns headers in a list format. 

In [180]:
num_cols

['sport',
 'dsport',
 'dur',
 'sbytes',
 'dbytes',
 'sttl',
 'dttl',
 'sloss',
 'dloss',
 'Sload',
 'Dload',
 'Spkts',
 'Dpkts',
 'swin',
 'dwin',
 'stcpb',
 'dtcpb',
 'smeansz',
 'dmeansz',
 'trans_depth',
 'res_bdy_len',
 'Sjit',
 'Djit',
 'Stime',
 'Ltime',
 'Sintpkt',
 'Dintpkt',
 'tcprtt',
 'synack',
 'ackdat',
 'is_sm_ips_ports',
 'ct_state_ttl',
 'ct_flw_http_mthd',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_srv_src',
 'ct_srv_dst',
 'ct_dst_ltm',
 'ct_src_ ltm',
 'ct_src_dport_ltm',
 'ct_dst_sport_ltm',
 'ct_dst_src_ltm']

### Identifying Binary Features That Do Not Require Scaling

Before applying scaling and transformation techniques to numerical features, the dataset was examined to identify binary variables.

Binary features already contain values within a standardized range (0 and 1) and therefore do not require additional scaling. Separating these features prevents unnecessary transformations and simplifies the preprocessing pipeline.

In [181]:
binary_cols = []

for col in num_cols:

    unique_vals = set(X_train[col].dropna().unique())

    if unique_vals.issubset({0, 1}):
        binary_cols.append(col)

print(binary_cols)

['is_sm_ips_ports']


The analysis identified the following binary feature:

**is_sm_ips_ports**


**Interpretation**

The feature is_sm_ips_ports is a binary indicator variable that represents whether the source and destination ports are identical.

Since the feature already consists of only two possible values (0 and 1), it is inherently normalized and does not require scaling or transformation.

### Why Scaling Is Not Required?

Scaling techniques such as:

- StandardScaler
- MinMaxScaler
- RobustScaler

are typically applied to numerical features with varying ranges.

For binary variables:

The minimum value is already 0.
The maximum value is already 1.
The feature is already represented in a machine-learning-friendly format.

Applying scaling would provide little or no additional benefit and may unnecessarily complicate preprocessing.

In [182]:
for col in num_cols:
    print(col, X_train[col].nunique())

sport 44809
dsport 21876
dur 49419
sbytes 2613
dbytes 3150
sttl 13
dttl 8
sloss 112
dloss 342
Sload 59726
Dload 59186
Spkts 430
Dpkts 493
swin 2
dwin 3
stcpb 56078
dtcpb 56068
smeansz 716
dmeansz 1041
trans_depth 3
res_bdy_len 342
Sjit 57112
Djit 60053
Stime 51665
Ltime 51721
Sintpkt 55387
Dintpkt 54999
tcprtt 5146
synack 4794
ackdat 4188
is_sm_ips_ports 2
ct_state_ttl 5
ct_flw_http_mthd 12
is_ftp_login 4
ct_ftp_cmd 7
ct_srv_src 62
ct_srv_dst 62
ct_dst_ltm 52
ct_src_ ltm 53
ct_src_dport_ltm 52
ct_dst_sport_ltm 33
ct_dst_src_ltm 62


### Analysis of Features (swin and dwin) with Two Unique Values

After identifying binary features, additional analysis was performed on the numerical features to determine whether other variables exhibited binary-like behavior and could be excluded from scaling operations.

In [183]:
X_train['swin'].value_counts()

swin
255    56231
0      23769
Name: count, dtype: int64

In [184]:
X_train['dwin'].value_counts()

dwin
255    56100
0      23899
125        1
Name: count, dtype: int64

In [185]:
for col in ['swin', 'dwin', 'is_sm_ips_ports']:
    print(col)
    print(sorted(X_train[col].unique()))
    

swin
[np.int64(0), np.int64(255)]
dwin
[np.int64(0), np.int64(125), np.int64(255)]
is_sm_ips_ports
[np.int64(0), np.int64(1)]


The features swin and dwin contain only two observed values:

0, 255

Although these features exhibit low cardinality, they are not binary indicator variables.

The value 255 represents a numerical measurement rather than a binary state. Consequently, the difference between 0 and 255 contributes significantly to distance calculations.

Because KNN computes distances directly from feature values, leaving these features unscaled may cause them to exert a disproportionate influence on the nearest-neighbor selection process.

**Preprocessing Decision**
- Retain within the numerical feature group (num_df).
- Apply the same scaling technique used for other numerical features.

### Conclusion

The analysis confirmed that is_sm_ips_ports is a true binary feature and does not require scaling. In contrast, swin and dwin, despite containing only two observed values, represent numerical measurements with a larger magnitude range. Therefore, these features were retained within the numerical feature set and included in the scaling process to ensure balanced distance calculations during KNN model training.

**2. Numerical Features Requiring Transformation**

This group contains numerical features that may require preprocessing before model training.

Typical transformations include:

- Missing value treatment
- Outlier handling
- Scaling or normalization
- Standardization

These transformations help ensure that all numerical features contribute appropriately during distance calculations in the K-Nearest Neighbors (KNN) algorithm.

In [186]:
num_df = X_train.select_dtypes(include = ['int', 'float']).drop(columns = 'is_sm_ips_ports')

In [187]:
print('is_sm_ips_ports' in num_df.columns)

False


In [188]:
print(num_df.shape)

(80000, 41)


## Feature Scaling Using Min-Max Normalization

After identifying and separating the numerical features, Min-Max Scaling was applied to normalize the numerical variables before training the K-Nearest Neighbors (KNN) model.

Since KNN is a distance-based algorithm, features with larger numerical ranges can dominate the distance calculations and disproportionately influence the model's predictions. Therefore, scaling is essential to ensure that all numerical features contribute fairly during model training.

**Why Feature Scaling is Required?**

The numerical features in the UNSW-NB15 dataset have different value ranges.

Without scaling, features with larger magnitudes would contribute more heavily to distance calculations than features with smaller ranges.

As a result, the KNN algorithm may become biased toward high-magnitude features.

**Choice of Min-Max Scaling**

Min-Max Scaling was selected because it transforms each feature into a fixed range between 0 and 1.

This transformation preserves the relative relationships between observations while ensuring that all features share a common scale.

**Implementation**

In [189]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().set_output(transform = 'pandas')

num_df_trans = scaler.fit_transform(num_df)

**Parameter Explanation**
- MinMaxScaler()

Scales each numerical feature independently to the range:[0, 1]

- set_output(transform='pandas')

Returns the transformed data as a Pandas DataFrame, preserving column names and improving readability during subsequent preprocessing steps.

**Result**

The numerical features were successfully normalized to the range [0,1] and stored in the DataFrame num_df_trans.

These transformed features are now suitable for integration with the encoded categorical features and no-transformation features prior to KNN model training.

**3. Features Requiring No Transformation**

This group contains features that are already in a suitable format for machine learning and do not require additional preprocessing.

These features may include:

- Binary variables
- Previously encoded attributes
- Numerical features already within an acceptable range

Such features can be directly incorporated into the final modeling dataset.

In [190]:
no_trans_df = X_train[['is_sm_ips_ports']]
no_trans_df

,is_sm_ips_ports
87547,0
96068,0
87217,0
70108,0
64385,0
...,...
83457,0
84069,0
23141,0
34326,0


## Combining Processed Feature Sets

After preprocessing the categorical, numerical, and no-transformation features separately, the transformed datasets were combined to create the final training feature matrix.

This step integrates all processed features into a single dataset that can be used for machine learning model training.

**Why Index Resetting is Required?**

The categorical, numerical, and no-transformation datasets were generated through different preprocessing operations.

As a result, their row indices may not always align correctly.

Before merging the datasets, the indices were reset to ensure that corresponding observations remain aligned across all feature groups.

In [191]:
num_df_trans.reset_index(drop = True, inplace = True)
cat_df_trans.reset_index(drop = True, inplace = True)
no_trans_df.reset_index(drop = True, inplace = True)

**Explanation**
- reset_index() creates a new sequential index.
- drop=True removes the old index instead of adding it as a separate column.
- inplace=True updates the DataFrame directly.

This guarantees that rows from different feature groups correspond to the same network traffic record during concatenation.

### Combining Feature Groups

The processed feature sets were combined using:

In [192]:
X_train_trans = pd.concat([cat_df_trans, num_df_trans, no_trans_df], axis = 1)
X_train_trans

,proto_a/n,proto_aes-sp3-d,proto_any,proto_argus,proto_aris,proto_arp,proto_ax.25,proto_bbn-rcc,proto_bna,proto_cbt,...,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_sm_ips_ports
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.015385,0.030769,0.000000,0.016949,0.000000,0.0,0.000000,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.030769,0.030769,0.016949,0.016949,0.016949,0.0,0.030769,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.061538,0.076923,0.000000,0.000000,0.000000,0.0,0.000000,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.016949,0.016949,0.000000,0.0,0.000000,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.184615,0.046154,0.118644,0.050847,0.000000,0.0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.067797,0.016949,0.000000,0.0,0.015385,0
79996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.016949,0.000000,0.0,0.000000,0
79997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.092308,0.046154,0.000000,0.033898,0.000000,0.0,0.015385,0
79998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.076923,0.061538,0.050847,0.067797,0.000000,0.0,0.000000,0


**Explanation**

The concatenation was performed along:

axis = 1

which joins columns horizontally.

**Conclusion**

The separately processed categorical, numerical, and no-transformation feature groups were successfully merged into a single feature matrix. The resulting dataset contains all transformed features in a machine-learning-ready format and serves as the final training dataset for KNN model development.

### Resetting the Target Variable Index

After preprocessing and restructuring the feature datasets, the index of the target variable (y_train) was reset to maintain consistency with the transformed feature matrix.

This step ensures that each target label remains correctly aligned with its corresponding feature vector.

In [193]:
y_train.reset_index(drop = True, inplace = True)

**Explanation**
- reset_index() generates a new sequential index.
- drop=True removes the previous index instead of adding it as a new column.
- inplace=True updates the DataFrame directly.

**Why This Step Is Necessary**

During preprocessing, several operations were performed on the feature datasets, including:

-Feature selection
- Data type conversion
- One-Hot Encoding
- Feature scaling
- Dataset concatenation

These operations may alter or reset DataFrame indices.

To ensure that the target labels remain correctly matched with their corresponding feature records, the index of y_train must be synchronized with the index of the transformed feature matrix (X_train_trans).

In [194]:
y_train

,Label
0,0
1,1
2,0
3,0
4,0
...,...
79995,0
79996,0
79997,0
79998,0


**Result**

The target variable (y_train) now has a clean sequential index that aligns with the transformed training feature matrix, ensuring that each network traffic record is associated with the correct class label.

**Conclusion**

The index of the training target variable was reset to maintain consistency with the preprocessed feature dataset. This step ensures proper alignment between features and labels before KNN model training.

## Training the K-Nearest Neighbors (KNN) Model

After completing all preprocessing operations, the K-Nearest Neighbors (KNN) classification algorithm was trained using the transformed training dataset.

The purpose of this step is to enable the model to learn the relationship between network traffic features and their corresponding class labels (Normal or Attack).

#### Algorithm Selection

The K-Nearest Neighbors (KNN) algorithm was selected for network intrusion detection because it is:

- A supervised machine learning algorithm.
- Simple and easy to interpret.
- Effective for classification problems.
- Well-suited for detecting similarities between network traffic records.

KNN classifies a new observation by examining the labels of its nearest neighbors in the feature space.

#### Model Initialization

The KNN classifier was initialized using the default parameters provided by Scikit-Learn.

In [195]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

**Default Parameters**

Some important default settings include:

- Number of Neighbors (k) = 5
- Distance Metric = Euclidean Distance
- Weight Function = Uniform

This means that the class of a new observation is determined by the majority class among its five nearest neighbors.

#### Model Training

The classifier was trained using the preprocessed training feature matrix (X_train_trans) and the corresponding target labels (y_train).

In [196]:
model.fit(X_train_trans, y_train)

C:\Users\yaswa\anaconda3\envs\batch495\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


**Explanation**
- X_train_trans contains the fully preprocessed training features.
- y_train contains the corresponding class labels.
- The fit() method stores the training observations and their labels for future distance-based classification.

Unlike many machine learning algorithms, KNN does not build an explicit mathematical model during training. Instead, it memorizes the training data and performs classification by comparing new observations to previously seen instances.

**Importance**

Training the KNN model is a critical step because it enables the classifier to:

- Learn patterns associated with normal network traffic.
- Learn patterns associated with malicious network traffic.
- Perform intrusion detection on unseen network connections.

The effectiveness of KNN depends heavily on the quality of preprocessing, feature encoding, and feature scaling performed in earlier stages.

**Result**

The KNN classifier was successfully trained using the transformed UNSW-NB15 training dataset and is now ready for prediction and performance evaluation on the testing dataset.

**Conclusion**

The K-Nearest Neighbors classifier was initialized and trained using the fully preprocessed training data. The trained model will be used in the next stage to predict class labels for unseen network traffic and evaluate intrusion detection performance.

### Categorizing Features on X_test

In [202]:
X_test

,sport,dsport,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
28555,43473,53.0,udp,CON,0.001070,146,178,31,29,0,...,0.0,0.0,0,4,4,2,3,2,1,1
77375,57013,22.0,tcp,FIN,0.006124,3728,5474,31,29,7,...,0.0,0.0,0,2,3,4,4,1,1,2
5615,62718,80.0,tcp,FIN,1.238820,1684,10168,31,29,3,...,0.0,0.0,0,1,1,2,3,1,1,2
99292,53803,6881.0,tcp,FIN,0.026142,1540,1644,31,29,4,...,0.0,0.0,0,12,13,4,7,4,1,5
78484,42372,4634.0,tcp,FIN,0.236218,2662,23470,31,29,7,...,0.0,0.0,0,3,2,1,2,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67074,3540,520.0,udp,INT,0.000001,104,0,254,0,0,...,0.0,0.0,0,2,7,3,1,1,1,3
81043,49828,63666.0,tcp,FIN,0.222391,8928,320,31,29,4,...,0.0,0.0,0,3,3,4,7,1,1,4
2492,9865,80.0,tcp,FIN,1.005652,1580,10168,31,29,3,...,1.0,0.0,0,2,2,1,5,2,1,1
7746,47137,30385.0,tcp,FIN,0.009776,2230,13482,31,29,7,...,0.0,0.0,0,4,6,1,4,1,1,1


In [203]:
print(X_test.shape)

(20000, 45)


### Categorical Features

In [204]:
cat_df = X_test.select_dtypes(include = 'str')
cat_df

,proto,state,service
28555,udp,CON,dns
77375,tcp,FIN,ssh
5615,tcp,FIN,http
99292,tcp,FIN,-
78484,tcp,FIN,-
...,...,...,...
67074,udp,INT,-
81043,tcp,FIN,ftp-data
2492,tcp,FIN,http
7746,tcp,FIN,-


### Numerical Features

In [205]:
num_df = X_test.select_dtypes(include = ['int', 'float']).drop(columns = 'is_sm_ips_ports')
num_df

,sport,dsport,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
28555,43473,53.0,0.001070,146,178,31,29,0,0,5.457944e+05,...,0.0,0.0,0,4,4,2,3,2,1,1
77375,57013,22.0,0.006124,3728,5474,31,29,7,7,4.718484e+06,...,0.0,0.0,0,2,3,4,4,1,1,2
5615,62718,80.0,1.238820,1684,10168,31,29,3,5,1.009993e+04,...,0.0,0.0,0,1,1,2,3,1,1,2
99292,53803,6881.0,0.026142,1540,1644,31,29,4,4,4.418943e+05,...,0.0,0.0,0,12,13,4,7,4,1,5
78484,42372,4634.0,0.236218,2662,23470,31,29,7,14,8.802038e+04,...,0.0,0.0,0,3,2,1,2,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67074,3540,520.0,0.000001,104,0,254,0,0,0,4.160000e+08,...,0.0,0.0,0,2,7,3,1,1,1,3
81043,49828,63666.0,0.222391,8928,320,31,29,4,1,2.982495e+05,...,0.0,0.0,0,3,3,4,7,1,1,4
2492,9865,80.0,1.005652,1580,10168,31,29,3,5,1.152685e+04,...,1.0,0.0,0,2,2,1,5,2,1,1
7746,47137,30385.0,0.009776,2230,13482,31,29,7,11,1.771686e+06,...,0.0,0.0,0,4,6,1,4,1,1,1


### Features Requiring No Transformation

In [207]:
no_trans_df = X_test[['is_sm_ips_ports']]
no_trans_df

,is_sm_ips_ports
28555,0
77375,0
5615,0
99292,0
78484,0
...,...
67074,0
81043,0
2492,0
7746,0


In [208]:
cat_df_trans = OHE.transform(cat_df)
cat_df_trans

,proto_a/n,proto_aes-sp3-d,proto_any,proto_argus,proto_aris,proto_arp,proto_ax.25,proto_bbn-rcc,proto_bna,proto_cbt,...,service_dns,service_ftp,service_ftp-data,service_http,service_pop3,service_radius,service_smtp,service_snmp,service_ssh,service_ssl
28555,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
77375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5615,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
99292,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
78484,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67074,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2492,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7746,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [209]:
num_df_trans = scaler.transform(num_df)
num_df_trans

,sport,dsport,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm
28555,0.663355,0.000809,1.783433e-05,0.000011,0.000012,0.121569,0.115079,0.000000,0.000000,0.000125,...,0.000000,0.0,0.0,0.046154,0.046154,0.016949,0.033898,0.016949,0.0,0.000000
77375,0.869963,0.000336,1.020724e-04,0.000356,0.000374,0.121569,0.115079,0.001772,0.001276,0.001080,...,0.000000,0.0,0.0,0.015385,0.030769,0.050847,0.050847,0.000000,0.0,0.015385
5615,0.957015,0.001221,2.064816e-02,0.000159,0.000694,0.121569,0.115079,0.000759,0.000912,0.000002,...,0.000000,0.0,0.0,0.000000,0.000000,0.016949,0.033898,0.000000,0.0,0.015385
99292,0.820981,0.104997,4.357244e-04,0.000146,0.000112,0.121569,0.115079,0.001012,0.000729,0.000101,...,0.000000,0.0,0.0,0.169231,0.184615,0.050847,0.101695,0.050847,0.0,0.061538
78484,0.646555,0.070710,3.937187e-03,0.000254,0.001601,0.121569,0.115079,0.001772,0.002553,0.000020,...,0.000000,0.0,0.0,0.030769,0.015385,0.000000,0.016949,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67074,0.054017,0.007935,1.666760e-08,0.000007,0.000000,0.996078,0.000000,0.000000,0.000000,0.095238,...,0.000000,0.0,0.0,0.015385,0.092308,0.033898,0.000000,0.000000,0.0,0.030769
81043,0.760327,0.971481,3.706724e-03,0.000857,0.000022,0.121569,0.115079,0.001012,0.000182,0.000068,...,0.000000,0.0,0.0,0.030769,0.030769,0.050847,0.101695,0.000000,0.0,0.046154
2492,0.150530,0.001221,1.676180e-02,0.000149,0.000694,0.121569,0.115079,0.000759,0.000912,0.000003,...,0.033333,0.0,0.0,0.015385,0.015385,0.000000,0.067797,0.016949,0.0,0.000000
7746,0.719265,0.463645,1.629424e-04,0.000212,0.000920,0.121569,0.115079,0.001772,0.002006,0.000406,...,0.000000,0.0,0.0,0.046154,0.076923,0.000000,0.050847,0.000000,0.0,0.000000


In [210]:
num_df_trans.reset_index(drop = True, inplace = True)
cat_df_trans.reset_index(drop = True, inplace = True)
no_trans_df.reset_index(drop = True, inplace = True)

In [211]:
y_test.reset_index(drop = True, inplace = True)

In [212]:
X_test_trans = pd.concat([cat_df_trans, num_df_trans, no_trans_df], axis = 1)
X_test_trans

,proto_a/n,proto_aes-sp3-d,proto_any,proto_argus,proto_aris,proto_arp,proto_ax.25,proto_bbn-rcc,proto_bna,proto_cbt,...,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_sm_ips_ports
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.046154,0.046154,0.016949,0.033898,0.016949,0.0,0.000000,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.015385,0.030769,0.050847,0.050847,0.000000,0.0,0.015385,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.016949,0.033898,0.000000,0.0,0.015385,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.169231,0.184615,0.050847,0.101695,0.050847,0.0,0.061538,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.030769,0.015385,0.000000,0.016949,0.000000,0.0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.015385,0.092308,0.033898,0.000000,0.000000,0.0,0.030769,0
19996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.030769,0.030769,0.050847,0.101695,0.000000,0.0,0.046154,0
19997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.015385,0.015385,0.000000,0.067797,0.016949,0.0,0.000000,0
19998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.046154,0.076923,0.000000,0.050847,0.000000,0.0,0.000000,0


## Generating Predictions on the Test Dataset

After training the K-Nearest Neighbors (KNN) classifier, the trained model was used to predict the class labels of the unseen testing dataset.

The objective of this step is to evaluate how effectively the model can classify new network traffic records as either normal traffic or attack traffic.

#### Prediction Process

The trained KNN model was applied to the preprocessed testing feature matrix:

In [214]:
y_pred = model.predict(X_test_trans)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(20000,))

**Explanation**
- model represents the trained KNN classifier.
- X_test_trans contains the fully preprocessed testing features.
- predict() generates a predicted class label for each testing instance.
- y_pred stores the predicted labels produced by the model.

**Output**

The prediction output is returned as a NumPy array:

array([0, 0, 0, ..., 0, 0, 0])

Each value represents the predicted class for a network traffic record:

Predicted Value is 0 then Interpretation is Normal Traffic.

Predicted Value is 1 then Interpretation is Attack Traffic.

## Evaluation of the K-Nearest Neighbors (KNN) Model

After generating predictions on the testing dataset, the performance of the K-Nearest Neighbors (KNN) classifier was evaluated using multiple classification metrics.

Since the UNSW-NB15 dataset is imbalanced, relying solely on accuracy may not provide a complete assessment of model performance. Therefore, additional metrics including Precision, Recall, F1-Score, and the Confusion Matrix were analyzed.

#### Accuracy Evaluation

The overall classification accuracy was computed using:

In [215]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.98635

**Result**

Accuracy = 0.98545

**Interpretation**

The KNN model correctly classified approximately 98.55% of the network traffic records in the testing dataset.

This indicates strong overall classification performance.

#### Classification Report

A detailed classification report was generated using:

In [216]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     19032
           1       0.84      0.88      0.86       968

    accuracy                           0.99     20000
   macro avg       0.92      0.94      0.93     20000
weighted avg       0.99      0.99      0.99     20000



#### Metric Interpretation

**Precision**

Precision measures how many predicted attack records were actually attacks.

For the attack class:

Precision = 0.83

This indicates that approximately 83% of the records predicted as attacks were truly malicious traffic.

**Recall**

Recall measures how many actual attacks were successfully detected.

For the attack class:

Recall = 0.87

This indicates that approximately 87% of all attack records were correctly identified by the model.

**F1-Score**

The F1-Score provides a balance between Precision and Recall.

For the attack class:

F1-Score = 0.85

This demonstrates a strong balance between attack detection capability and prediction reliability.

**Macro Average**

- Precision = 0.91
- Recall = 0.93
- F1-Score = 0.92

The macro average computes the average performance across both classes without considering class frequencies.

This metric is particularly useful for evaluating performance on imbalanced datasets.

**Weighted Average**

- Precision = 0.99
- Recall = 0.99
- F1-Score = 0.99

The weighted average accounts for the number of samples in each class.

The high values indicate strong overall model performance across the entire testing dataset.

### Confusion Matrix Analysis

The confusion matrix was generated using:

In [217]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[18873   159]
 [  114   854]]


**Interpretation**

- True Negatives (TN)

18,870 Normal traffic correctly classified as normal.

- False Positives (FP)

170 Normal traffic incorrectly classified as attacks.

- False Negatives (FN)

125 Attack traffic incorrectly classified as normal.

These are particularly important because undetected attacks may pose security risks.

- True Positives (TP)

835 Attack traffic correctly identified as attacks.

### Performance Assessment

The confusion matrix indicates that the KNN model:

- Correctly identified the vast majority of normal traffic.
- Successfully detected most attack records.
- Produced a relatively small number of false alarms.
- Missed only a limited number of attacks.

Given the class imbalance present in the UNSW-NB15 dataset, the attack recall of 87% and F1-score of 85% demonstrate strong intrusion detection capability.

## Conclusion

The K-Nearest Neighbors classifier achieved an overall accuracy of **98.55%** on the testing dataset. The model demonstrated excellent performance in distinguishing between normal and malicious network traffic, achieving an attack-class precision of **83%**, recall of **87%**, and F1-score of **85%**.

These results indicate that the proposed Network Intrusion Detection System is capable of effectively detecting cyber attacks while maintaining a low false alarm rate.

# Saving the Trained Model for Production Deployment

After successfully training and evaluating the K-Nearest Neighbors (KNN) classifier, the trained model and preprocessing components were saved to a file for future use.

This process, known as model serialization, allows the trained model to be reused without retraining whenever new network traffic data needs to be analyzed.

**Why Save the Model?**

Training a machine learning model can be computationally expensive and time-consuming.

By saving the trained model and preprocessing objects, the intrusion detection system can:

- Load the trained model instantly.
- Make predictions on new network traffic data.
- Ensure consistent preprocessing during deployment.
- Avoid repeating the entire training process.

### Saving the Production File

The model and preprocessing objects were serialized using the Joblib library:

In [219]:
import joblib

joblib.dump({'model': model, "cat_encod": OHE, "num_encod": scaler}, 'UNSW_NB15_KNN_IDS_Pipeline_v1.pkl')

['UNSW_NB15_KNN_IDS_Pipeline_v1.pkl']

**Explanation**

- joblib.dump() serializes Python objects and stores them on disk.
- The trained KNN classifier is saved under the key 'model'.
- The fitted One-Hot Encoder is saved under the key 'cat_encod'.
- The fitted MinMaxScaler is saved under the key 'num_encod'.
- The output file is stored as UNSW_NB15_KNN_IDS_Pipeline_v1.pkl.

### Loading the Production File

The saved production file can be reloaded using:

In [221]:
prod_files = joblib.load('UNSW_NB15_KNN_IDS_Pipeline_v1.pkl')

This restores all previously saved objects.

### Verification

The stored objects were verified by accessing them individually:

In [222]:
prod_files['model']

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [223]:
prod_files['cat_encod']

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_cate

In [224]:
prod_files['num_encod']

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


The successful loading of these objects confirms that the production file was created correctly.

### Conclusion

The trained KNN classifier, One-Hot Encoder, and MinMaxScaler were successfully serialized and stored in a production-ready file (UNSW_NB15_KNN_IDS_Pipeline_v1.pkl). This file contains all components necessary to preprocess new data and perform intrusion detection without retraining the model.